# Problem Statement

The goal of this project is to predict whether a heart failure patient will be readmitted to the hospital within 30 days.

Hospital readmissions are a major challenge in healthcare because they increase costs and indicate poor patient outcomes.

We use machine learning to analyze both clinical and social factors such as:
- Vital signs (blood pressure, heart rate)
- Clinical markers (BNP, creatinine, sodium)
- Medication and treatment patterns
- Social factors (income level, access to care, medication adherence)

The objective is to build a predictive model that identifies high-risk patients early and helps reduce hospital readmissions.

#### Load dataset and import libbraries 

In [ ]:
# Import Libraries for handling Data

import pandas as pd
import numpy as np

# Import train-test split
from sklearn.model_selection import train_test_split

# Import Logistic Regression
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Import SMOTE for handling class imbalance
from imblearn.over_sampling import SMOTE

# Import Random Forest model
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler

# Import SVM model
from sklearn.svm import SVC

# Import KNN model
from sklearn.neighbors import KNeighborsClassifier

# Import Naive Bayes model
from sklearn.naive_bayes import GaussianNB

In [ ]:
# Load Datasets

df = pd.read_csv('heart_failure_readmission_dataset.csv')

# Data Head

df.head()


In [ ]:
# Check dataset shape (rows, columns)

print('Shape of Dataset:',df.shape)

# Check column names 
print("\nColumns:\n", df.columns)

# chedck for Missing values

print("\nMissing values:\n", df.isnull().sum())

# Basic infor about data types
print("\nData info:")
df.info()

### Lets Handle missing values (ONLY missing values first)

We saw:

- bmi → 90 missing
- sodium → 90 missing
- creatinine → 90 missing

#### 🧠 Why median?

- Medical data has outliers
- Median is more stable than mean
- Prevents distortion of data


In [ ]:
# Fill missing vlues with median (Safe for medical numerical data)

df['bmi'] =df['bmi'].fillna(df['bmi'].median())
df["sodium"] = df["sodium"].fillna(df["sodium"].median())
df["creatinine"] = df["creatinine"].fillna(df["creatinine"].median())

#### 🟢  Encode categorical columns

Now we convert text into numbers so the model can understand it.

In [ ]:
# Convert gender into numbers

df['gender'] = df['gender'].map({"Male": 1, "Female": 0})

# Convert income level into numbers

df['income_level'] = df['income_level'].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

#### 🟢 Step 4.3: Final Data Check (Before Modeling)

In [ ]:
# Check first 5 rows
df.head()

In [ ]:
# Check data type

df.info()

In [ ]:
# Check if any missing values still exist

df.isnull().sum()

#### 🟢  Train/Test Split
Now we separate:

- features (X)
- target (y)

In [ ]:
X= df.drop("readmitted_30d", axis=1)
y = df['readmitted_30d']

In [ ]:
# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Check shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

#### 🟢  Logistic Regression (Baseline Model)

Now we build your first ML model.

In [ ]:
# Create model

log_model = LogisticRegression(max_iter=1000)

# Train model

log_model.fit(X_train, y_train)

#### Make predictions

In [ ]:
y_pred =log_model.predict(X_test)

#### Evaluate Logistic Regression Model

In [ ]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Accuracy
print('\nAccuracy Score:')
print(accuracy_score(y_test, y_pred))

##### 🟢 Model Evaluation Findings (Logistic Regression)

- Overall Accuracy = 65%
- The model is correct 65 out of 100 times
- This is moderate performance
- Not strong enough for healthcare use yet

##### 📊 2. Confusion Matrix Meaning

[[292  82]
 [128  98]]

 - 292 (True Negatives)
→ Correctly predicted patients who will NOT be readmitted
- 82 (False Positives)
→ Model wrongly predicted readmission (alarm but wrong)
- 128 (False Negatives) 🚨
→ Model FAILED to detect real readmitted patients
- 98 (True Positives)
→ Correctly detected readmitted patients

##### Most Important Problem

👉 The model is missing many sick patients

- Recall for class 1 = 0.43

##### This means:
- It only detects 43% of real readmissions
- It misses 57% of high-risk patients

##### Problem Class Imbalance Impact

- Class 0 (no readmission) = majority
- Class 1 (readmission) = minority

###### 👉 So the model learns:

“It is safer to predict 0 most of the time”

## Apply SMOTE (Balance Training Data Only)

In [ ]:
# Create SMOTE object

smote = SMOTE(random_state=42)

# Apply SMOTE only on training data

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

#### Check New Class Balance

In [ ]:
# Check the new balanced class distribution

print(y_train_smote.value_counts())

- D ataset is now fully balanced
- Both classes have equal importance
- The model will no longer “prefer” class 0
- This should improve recall for class 1 (readmitted patients)

##### Train Logistic Regression (SMOTE data)

In [ ]:
# Create model

log_model_smote = LogisticRegression(max_iter=1000)

# Train using SMOTE data

log_model_smote.fit(X_train_smote, y_train_smote)


####  Predict on original test set

In [ ]:
# Predict on original test set
y_pred_smote = log_model_smote.predict(X_test)

##### Evaluate SMOTE Logistic Regression Model

In [ ]:
# Confusion Matrix

print('Confusion Matrix')
print(confusion_matrix(y_test, y_pred_smote))

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

In [ ]:
# Accuracy Score
print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_smote))

##### 🟢 SMOTE Model Evaluation (Simple Explanation)

###### 📊 Confusion Matrix

[[234 140]
 [ 90 136]]
 
 Class 0 (Not readmitted)
- 234 correctly predicted
- 140 incorrectly predicted

👉 Model is still okay but slightly weaker than before

Class 1 (Readmitted) 🚨 IMPORTANT
- 136 correctly detected (GOOD ✔)
- 90 missed (still errors, but improved)

###### 📈 Key Improvements (VERY IMPORTANT)
Before SMOTE:
- Recall for class 1 = 0.43
- Model was missing most readmitted patients
After SMOTE:
- Recall for class 1 = 0.60 ✔

👉 That is a BIG improvement

###### 🧠 Final Interpretation

- Model now detects more high-risk patients
- Trade-off: slightly more false alarms
- Overall performance is more balanced

###### 📊 Accuracy = 61.6%

Lower than before (expected)
BUT more realistic and useful in healthcare

👉 In medical problems, recall is more important than accuracy



# Train Random Forest Model

We will now build a stronger model and compare it with Logistic Regression.

In [ ]:
# Create model

rf_model = RandomForestClassifier(random_state=42)

# Train model on SMOTE data 

rf_model.fit(X_train_smote, y_train_smote)

In [ ]:
# Make prediction

y_pred_rf = rf_model.predict(X_test)

## Evaluate Random Forest (and compare with Logistic Regression 🚀)

In [ ]:
# Confused Matrix
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
# Classification Report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Accuracy Score

print("\nAccuracy Score:")
print(accuracy_score(y_test,y_pred_rf))

### 🟢 Logistic Regression (SMOTE) vs Random Forest


| Metric              | Logistic Regression | Random Forest |
| ------------------- | ------------------- | ------------- |
| Accuracy            | 61.7%               | 64.3%         |
| Recall (Class 1)    | 0.60                | 0.58          |
| Precision (Class 1) | 0.49                | 0.52          |
| F1-score (Class 1)  | 0.54                | 0.55          |

### Accuracy improved

- Logistic Regression: 61.7%
- Random Forest: 64.3%

Random Forest makes more correct predictions overall.

### ✔ Precision improved

- Logistic Regression: 0.49
- Random Forest: 0.52

This means Random Forest produces fewer false alarms.

### ✔ F1-score improved

- Logistic Regression: 0.54
- Random Forest: 0.55

A small improvement in overall balance between precision and recall.



# Train XGBoost Model

In [ ]:
# Create XGBoost model

xgb_model = XGBClassifier(random_state=42,eval_metric="logloss")

# Train model using SMOTE-balanced data
xgb_model.fit(X_train_smote, y_train_smote)


In [ ]:
# Make predictions

y_pred_xgb =xgb_model.predict(X_test)

## Evaluate XGBoost Model

In [ ]:
# Confuion Matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_xgb))

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test,y_pred_xgb))

In [ ]:
# Accuracy Score

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_xgb))

## 🚀 XGBoost (SMOTE)

- Accuracy: 61.5%
- Recall (Class 1): 0.51
Key Findings:
- XGBoost did not outperform the other models on this dataset.
- It shows moderate predictive ability.
- It misses more readmitted patients compared to other models.
- Requires further tuning to improve performance.

## 🧠 Overall Conclusion

- SMOTE significantly improved model performance on the minority class.
- There is a clear trade-off between accuracy and recall.
- Logistic Regression performed best for identifying high-risk patients.
- Random Forest performed best in overall balance.
- XGBoost was not the best fit for this dataset without tuning.


# Let Us improve more by Feature Scaling

In [ ]:
scaler = StandardScaler()

# Fit only on training data

X_train_scaled = scaler.fit_transform(X_train_smote)

# Transform test data
X_test_scaled = scaler.transform(X_test)

## Support Vector Machine (SVM)

Because it is strong for classification problems.

In [ ]:
# Create SVM model
svm_model = SVC()

# Train on scaled SMOTE data
svm_model.fit(X_train_scaled, y_train_smote)

In [ ]:
# Make predictions
y_pred_svm = svm_model.predict(X_test_scaled)

In [ ]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_svm))

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test,y_pred_svm))

In [ ]:
# Accuracy Score
print('\nAccuracy Score:')
print(accuracy_score(y_test, y_pred_svm))

#### 🟢 SVM Model Evaluation 
[[253 121]
 [ 85 141]]

### 🧠 What this means

✔ Class 0 (Not readmitted)
- 253 correct predictions
- 121 wrong predictions

👉 Model is fairly good at identifying stable patients

### Class 1 (Readmitted patients 🚨)

- 141 correctly detected ✔ (GOOD improvement)
- 85 missed ❌ (still some errors)

👉 This is important improvement in recall

## K-Nearest Neighbors (KNN)
KNN is a distance-based model, so it works well with your scaled data ✔



In [ ]:
# Create KNN model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train model
knn_model.fit(X_train_scaled, y_train_smote)

In [ ]:
# Make predictions
y_pred_knn = knn_model.predict(X_test_scaled)

In [ ]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

In [ ]:
# Accuracy Score
print("\nAccurcy Score:")
print(accuracy_score(y_test, y_pred_knn))

## 🟢 KNN Model Evaluation

[[241 133]
 [ 93 133]]

 ### 🧠 What this means

✔ Class 0 (Not readmitted)
- 241 correct predictions
- 133 wrong predictions

👉 Model struggles a bit with stable patients


✔ Class 1 (Readmitted 🚨)
- 133 correctly detected ✔
- 93 missed ❌

👉 It catches moderate number of high-risk patients



| Model               | Accuracy | Recall (Class 1) | Comment             |
| ------------------- | -------- | ---------------- | ------------------- |
| Logistic Regression | 61%      | 0.60             | Good recall         |
| Random Forest       | 64%      | 0.58             | Balanced            |
| XGBoost             | 61%      | 0.51             | Weakest             |
| SVM                 | 66%      | 0.62 ⭐           | Best overall        |
| KNN                 | 62%      | 0.59             | Average performance |


# Let us See Naive Bayes Model



In [ ]:
# Create model
nb_model = GaussianNB()

# Train model (use SMOTE + scaled data? No scaling needed but OK to keep consistent)
nb_model.fit(X_train_smote, y_train_smote)

# Make predictions (use original test set, NOT scaled)
y_pred_nb = nb_model.predict(X_test)

In [ ]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

In [ ]:

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Accuracy Score
print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_nb))

# Naive Bayes Evaluation

🧠 What this means

- ✔ Class 0 (Not readmitted)
- 247 correct predictions
- 127 wrong predictions

👉 Model is fairly stable for non-readmitted patients

- Class 1 (Readmitted 🚨)
- 145 correctly detected ✔ (GOOD)
- 81 missed ❌ (better than many models)

👉 This is actually a strong recall improvement

| Model               | Accuracy | Recall (Class 1) | Key Insight        |
| ------------------- | -------- | ---------------- | ------------------ |
| Logistic Regression | 61%      | 0.60             | Good recall        |
| Random Forest       | 64%      | 0.58             | Balanced           |
| XGBoost             | 61%      | 0.51             | Weak               |
| SVM                 | 66%      | 0.62             | Best overall model |
| KNN                 | 62%      | 0.59             | Average            |
| **Naive Bayes**     | **65%**  | **0.64 ⭐**       | Best recall so far |
